<a href="https://colab.research.google.com/github/shankar-011/GenAI-Internship/blob/master/Day7/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv",encoding='latin1', on_bad_lines='skip', engine='python')
print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    16215
negative    16210
Name: count, dtype: int64


In [12]:
df['sentiment']=df['sentiment'].map({'positive':1,'negative':0})

In [13]:
from numpy import divide
import re
negation_words=[
    "not good","not bad","not great","don't like","didn't like","never liked","wasn't good","isn't good","no good"
]

def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-zA-Z\s']","",text)

  #convert negations into single tokens
  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [14]:
df['review']=df['review'].apply(clean_text)

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42)

In [16]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
vocab_size=20000
max_len=250
tokenizer=Tokenizer(num_words=vocab_size,oov_token="<00V") #out of vocabulary
tokenizer.fit_on_texts(X_train)
X_train_seq=tokenizer.texts_to_sequences(X_train)
X_test_seq=tokenizer.texts_to_sequences(X_test)
X_train_pad=pad_sequences(X_train_seq,maxlen=max_len,padding='post') #highest len onum chyilla,backi ellam zeroes itt adjust chyum.post means backil zeroes.
X_test_pad=pad_sequences(X_test_seq,maxlen=max_len,padding='post')

In [17]:
#LSTM(long short term memory)-advanced type of RNN
#forget gate-what to forget
#input gate-what to store
#cell state-in memory
#output gate-what to return
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout
model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),
    LSTM(128,dropout=0.3,recurrent_dropout=0.3),
    Dense(64,activation='relu'),
    Dropout(0.3),
    Dense(1,activation='sigmoid')
])
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [18]:
history=model.fit(X_train_pad,y_train,epochs=5,batch_size=64,validation_split=0.2)

Epoch 1/5
325/325 ━━━━━━━━━━━━━━━━━━━━ 301s 911ms/step - accuracy: 0.5412 - loss: 0.6834 - val_accuracy: 0.5362 - val_loss: 0.6748
Epoch 2/5
325/325 ━━━━━━━━━━━━━━━━━━━━ 323s 915ms/step - accuracy: 0.5861 - loss: 0.6334 - val_accuracy: 0.5098 - val_loss: 0.6928
Epoch 3/5
325/325 ━━━━━━━━━━━━━━━━━━━━ 289s 890ms/step - accuracy: 0.6026 - loss: 0.5921 - val_accuracy: 0.5619 - val_loss: 0.6663
Epoch 4/5
325/325 ━━━━━━━━━━━━━━━━━━━━ 289s 889ms/step - accuracy: 0.6276 - loss: 0.5530 - val_accuracy: 0.5806 - val_loss: 0.6866
Epoch 5/5
325/325 ━━━━━━━━━━━━━━━━━━━━ 295s 908ms/step - accuracy: 0.6952 - loss: 0.5000 - val_accuracy: 0.7697 - val_loss: 0.5779


In [20]:
loss,acc=model.evaluate(X_test_pad,y_test)
print("Test accuracy:",acc)

203/203 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - accuracy: 0.8290 - loss: 0.4586
Test accuracy: 0.8289899826049805


In [21]:
def predict_sentiment(review):
  review=clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded=pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview",review)
  print("Score:",prediction)
  if prediction>=0.5:
    print("sentiment: Positive")
  else:
    print("sentiment: Negative")

In [22]:
predict_sentiment("This movie was absolutely amazing and I loved it very much")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 485ms/step

Review this movie was absolutely amazing and i loved it very much
Score: 0.8595621
sentiment: Positive


In [23]:
predict_sentiment("This movie was not good and I didn't like it at all")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step

Review this movie was not_good and i didn't_like it at all
Score: 0.6394855
sentiment: Positive


In [24]:
predict_sentiment("This movie is very entertaining! ")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step

Review this movie is very entertaining 
Score: 0.8015864
sentiment: Positive
